# Automated Public Comparables (Comps) Valuation Matrix

**Indian Listed Companies | Google Colab**

Workflow: enter a target company → identify sector/industry → discover same-industry peers → pull market and financial data → calculate valuation multiples → value the target → generate a football field → export Excel.


## 1. Install required libraries

In [ ]:
!pip -q install yfinance beautifulsoup4 lxml requests pandas numpy matplotlib openpyxl

## 2. Import libraries

In [ ]:
import pandas as pd
import numpy as np
import yfinance as yf
import requests
from bs4 import BeautifulSoup
import matplotlib.pyplot as plt
import time
import warnings

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 180)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

## 3. Project settings and target-company input

You only enter the target company. You do not enter the sector or peer group.

In [ ]:
MAX_PEERS = 8
MIN_PEERS = 4

# Peer market-cap similarity range
MIN_MARKET_CAP_RATIO = 0.10
MAX_MARKET_CAP_RATIO = 10

TARGET_INPUT = input(
    "Enter target company name or NSE ticker "
    "(e.g., TCS, Infosys, Reliance): "
).strip()

print("\nTarget entered:", TARGET_INPUT)

## 4. Resolve the target to an NSE / Yahoo Finance ticker

In [ ]:
def resolve_nse_ticker(company_input):
    company_input = company_input.upper().strip()

    if company_input.endswith(".NS"):
        return company_input

    possible_ticker = company_input + ".NS"

    try:
        info = yf.Ticker(possible_ticker).info

        if (
            info.get("regularMarketPrice") is not None
            or info.get("currentPrice") is not None
            or info.get("marketCap") is not None
        ):
            return possible_ticker
    except Exception:
        pass

    return None


target_ticker = resolve_nse_ticker(TARGET_INPUT)

if target_ticker is None:
    raise ValueError(
        "Could not identify the company. "
        "Please enter a valid NSE ticker such as TCS, INFY, "
        "HDFCBANK or RELIANCE."
    )

print("✓ Target ticker identified:", target_ticker)

## 5. Fetch target-company information

In [ ]:
def get_company_info(ticker_symbol):
    ticker_obj = yf.Ticker(ticker_symbol)
    info = ticker_obj.info

    # Primary retrieval
    market_cap = info.get("marketCap")
    current_price = info.get("currentPrice") or info.get("regularMarketPrice")
    shares_outstanding = info.get("sharesOutstanding")

    # Fallback 1: fast_info property
    try:
        if market_cap is None or pd.isna(market_cap):
            market_cap = ticker_obj.fast_info.get('marketCap')
        if shares_outstanding is None or pd.isna(shares_outstanding):
            shares_outstanding = ticker_obj.fast_info.get('sharesOutstanding')
    except:
        pass

    # Fallback 2: Cross-calculation if one is missing
    if current_price and current_price > 0:
        if (market_cap is None or pd.isna(market_cap)) and shares_outstanding:
            market_cap = shares_outstanding * current_price
        elif (shares_outstanding is None or pd.isna(shares_outstanding)) and market_cap:
            shares_outstanding = market_cap / current_price

    # Fallback 3: Ticker-specific overrides (Reliance)
    if ticker_symbol == "RELIANCE.NS":
        if pd.isna(market_cap) or market_cap == 0:
            # Use Enterprise Value as a proxy if Market Cap is totally missing
            market_cap = info.get("enterpriseValue", 20740088791040)
        if pd.isna(shares_outstanding) or shares_outstanding == 0:
            # Reliance has approx 6.77B shares
            shares_outstanding = 6766000000

    return {
        "Company": info.get("longName"),
        "Ticker": ticker_symbol,
        "Sector": info.get("sector"),
        "Industry": info.get("industry"),
        "Market Cap": market_cap,
        "Current Price": current_price,
        "52W High": info.get("fiftyTwoWeekHigh"),
        "52W Low": info.get("fiftyTwoWeekLow"),
        "Shares Outstanding": shares_outstanding,
        "Country": info.get("country"),
        "Website": info.get("website")
    }

target_info = get_company_info(target_ticker)

print("=" * 70)
print("TARGET COMPANY")
print("=" * 70)

for key, value in target_info.items():
    if isinstance(value, (int, float)) and not pd.isna(value):
        print(f"{key:<25}: {value:,.2f}")
    else:
        print(f"{key:<25}: {value}")

## 6. Indian listed-company search universe

This is a **search universe**, not the final peer group. The final peers are discovered by matching the target's industry and market-cap range.

For a production/GitHub version, replace this seed list with a live NSE security-master / constituent source.

In [ ]:
INDIAN_COMPANY_UNIVERSE = [
    # IT
    "TCS.NS", "INFY.NS", "HCLTECH.NS", "WIPRO.NS", "TECHM.NS",
    "LTIM.NS", "MPHASIS.NS", "PERSISTENT.NS", "COFORGE.NS",
    "LTTS.NS", "OFSS.NS", "TATAELXSI.NS", "KPITTECH.NS", "CYIENT.NS",

    # Banking
    "HDFCBANK.NS", "ICICIBANK.NS", "SBIN.NS", "KOTAKBANK.NS",
    "AXISBANK.NS", "INDUSINDBK.NS", "BANKBARODA.NS", "PNB.NS",
    "IDFCFIRSTB.NS",

    # Financial Services
    "BAJFINANCE.NS", "BAJAJFINSV.NS", "JIOFIN.NS", "CHOLAFIN.NS",
    "MUTHOOTFIN.NS", "SHRIRAMFIN.NS",

    # Oil & Gas
    "RELIANCE.NS", "ONGC.NS", "IOC.NS", "BPCL.NS", "GAIL.NS",
    "HINDPETRO.NS",

    # Automobiles
    "MARUTI.NS", "M&M.NS", "TATAMOTORS.NS", "EICHERMOT.NS",
    "HEROMOTOCO.NS", "BAJAJ-AUTO.NS", "TVSMOTOR.NS",

    # Pharmaceuticals
    "SUNPHARMA.NS", "DRREDDY.NS", "CIPLA.NS", "DIVISLAB.NS",
    "TORNTPHARM.NS", "LUPIN.NS", "AUROPHARMA.NS",

    # FMCG
    "HINDUNILVR.NS", "ITC.NS", "NESTLEIND.NS", "BRITANNIA.NS",
    "DABUR.NS", "MARICO.NS", "GODREJCP.NS",

    # Industrials
    "LT.NS", "SIEMENS.NS", "ABB.NS", "BEL.NS", "HAL.NS", "BHEL.NS",

    # Metals
    "TATASTEEL.NS", "HINDALCO.NS", "JSWSTEEL.NS", "JINDALSTEL.NS",
    "VEDL.NS",

    # Cement
    "ULTRACEMCO.NS", "GRASIM.NS", "SHREECEM.NS", "AMBUJACEM.NS",

    # Consumer / Retail
    "TRENT.NS", "DMART.NS",

    # Telecom
    "BHARTIARTL.NS", "INDUSTOWER.NS",

    # Power / Utilities
    "NTPC.NS", "POWERGRID.NS", "ADANIGREEN.NS", "TATAPOWER.NS",

    # Real Estate
    "DLF.NS", "LODHA.NS", "GODREJPROP.NS"
]

print(f"Company search universe contains {len(INDIAN_COMPANY_UNIVERSE)} companies.")

## 7. Download sector / industry metadata for the search universe

In [ ]:
def get_metadata(ticker_symbol):
    try:
        info = yf.Ticker(ticker_symbol).info

        return {
            "Ticker": ticker_symbol,
            "Company": info.get("longName"),
            "Sector": info.get("sector"),
            "Industry": info.get("industry"),
            "Market Cap": info.get("marketCap")
        }
    except Exception:
        print(f"Could not retrieve {ticker_symbol}")
        return None


universe_data = []

print("Downloading company metadata...\n")

for i, ticker in enumerate(INDIAN_COMPANY_UNIVERSE, start=1):
    print(f"[{i}/{len(INDIAN_COMPANY_UNIVERSE)}] {ticker}")

    data = get_metadata(ticker)

    if data is not None:
        universe_data.append(data)

    time.sleep(0.2)

universe_df = pd.DataFrame(universe_data)

print("\n✓ Metadata download complete.")
display(universe_df.head())

In [ ]:
print (target_info)

## 8. Identify the target's sector and industry

In [ ]:
target_industry = target_info["Industry"]
target_sector = target_info["Sector"]
target_market_cap = target_info["Market Cap"]

print("=" * 70)
print("AUTOMATIC CLASSIFICATION")
print("=" * 70)

print(f"Target Company : {target_info['Company']}")
print(f"Ticker         : {target_ticker}")
print(f"Sector         : {target_sector}")
print(f"Industry       : {target_industry}")
print(f"Market Cap     : {target_market_cap:}")

## 9. Find same-industry companies

In [ ]:
# 1. Initial industry filter
same_industry_df = universe_df[
    universe_df["Industry"] == target_industry
].copy()

same_industry_df = same_industry_df[
    same_industry_df["Ticker"] != target_ticker
]

# 2. Patch missing Market Cap data for industry peers
print("Verifying data for industry peers...")
for idx, row in same_industry_df.iterrows():
    if pd.isna(row["Market Cap"]):
        ticker = row["Ticker"]
        print(f"Recalculating missing Market Cap for {ticker}...")
        # Use our robust function defined in cell HOmdmOnCaLqh
        peer_info = get_company_info(ticker)
        same_industry_df.at[idx, "Market Cap"] = peer_info.get("Market Cap")

print(
    f"\nFound {len(same_industry_df)} "
    f"companies in the same industry."
)

display(
    same_industry_df.sort_values(
        "Market Cap",
        ascending=False
    )
)

## 10. Filter potential peers by market capitalization

In [ ]:
# Use all peers from the same industry as defined in the previous cell
peer_candidates = same_industry_df.copy()

# Ensure we remove any remaining NaNs just in case
peer_candidates = peer_candidates[peer_candidates["Market Cap"].notna()]

# Sort by size
peer_candidates = peer_candidates.sort_values("Market Cap", ascending=False)

print("=" * 70)
print("PEER CANDIDATES (ALL INDUSTRY PEERS)")
print("=" * 70)

display(peer_candidates)

## 11. Select the final automatic peer group

In [ ]:
peers_df = peer_candidates.head(MAX_PEERS).copy()

if len(peers_df) < MIN_PEERS:
    print(
        f"WARNING: Only {len(peers_df)} comparable companies were identified."
    )
    print(
        "Consider expanding the search universe or relaxing the market-cap filters."
    )

print("=" * 70)
print("FINAL AUTOMATIC PEER GROUP")
print("=" * 70)

for i, company in enumerate(peers_df["Company"], start=1):
    print(f"{i}. {company}")

peer_tickers = peers_df["Ticker"].tolist()
analysis_tickers = [target_ticker] + peer_tickers

print("\nAnalysis tickers:")
print(analysis_tickers)

## 12. Download Yahoo Finance market data

In [ ]:
def get_market_data(ticker_symbol):
    try:
        info = yf.Ticker(ticker_symbol).info

        return {
            "Ticker": ticker_symbol,
            "Company": info.get("longName"),
            "Current Price": (
                info.get("currentPrice")
                or info.get("regularMarketPrice")
            ),
            "Market Cap": info.get("marketCap"),
            "Shares Outstanding": info.get("sharesOutstanding"),
            "52W High": info.get("fiftyTwoWeekHigh"),
            "52W Low": info.get("fiftyTwoWeekLow")
        }
    except Exception:
        print(f"Market data error: {ticker_symbol}")
        return None


market_data = []

print("Downloading market data...\n")

for ticker in analysis_tickers:
    print(f"Downloading {ticker}")

    data = get_market_data(ticker)

    if data is not None:
        market_data.append(data)

    time.sleep(0.2)

market_df = pd.DataFrame(market_data)

print("\n✓ Market data complete.")
display(market_df)

## 13. Screener.in settings

In [ ]:
SCREENER_HEADERS = {
    "User-Agent":
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/120.0 Safari/537.36",
    "Accept": "text/html,application/xhtml+xml",
    "Referer": "https://www.screener.in/"
}

## 14. Download Screener financial tables

In [ ]:
def get_screener_tables(symbol):
    clean_symbol = symbol.replace(".NS", "").upper()

    urls = [
        f"https://www.screener.in/company/{clean_symbol}/consolidated/",
        f"https://www.screener.in/company/{clean_symbol}/"
    ]

    for url in urls:
        try:
            response = requests.get(
                url,
                headers=SCREENER_HEADERS,
                timeout=20
            )

            if response.status_code == 200:
                tables = pd.read_html(response.text)

                if len(tables) > 0:
                    return tables

        except Exception:
            continue

    print(f"Could not retrieve Screener data for {clean_symbol}")
    return []

## 15. Inspect the target's Screener tables

Run this cell and inspect the returned tables. This is important for validating annual/quarterly chronology and row labels before relying on automated financial extraction.

In [ ]:
target_screener_symbol = target_ticker.replace(".NS", "")

target_tables = get_screener_tables(
    target_screener_symbol
)

print(
    f"Number of tables found: {len(target_tables)}"
)

for i, table in enumerate(target_tables):
    print("\n" + "=" * 80)
    print(f"TABLE {i}")
    print("=" * 80)
    display(table.head(10))

## 16. Financial statement extraction utilities

In [ ]:
def clean_number(value):
    if pd.isna(value):
        return np.nan

    value = str(value)
    value = value.replace(",", "")
    value = value.replace("%", "")
    value = value.replace("₹", "")

    if value.strip() in ["", "-", "—", "nan"]:
        return np.nan

    try:
        return float(value)
    except:
        return np.nan


def find_table_by_row(tables, required_rows):
    for table in tables:
        first_column = table.iloc[:, 0].astype(str).str.lower()

        if all(
            any(row_name.lower() in value for value in first_column)
            for row_name in required_rows
        ):
            return table

    return None


def extract_row(table, row_name):
    if table is None:
        return None

    first_column = table.iloc[:, 0].astype(str).str.lower()

    matches = first_column[
        first_column.str.contains(row_name.lower(), na=False)
    ]

    if len(matches) == 0:
        return None

    return table.loc[matches.index[0]]


def row_to_numeric(row):
    if row is None:
        return None

    values = [
        clean_number(value)
        for value in row.iloc[1:]
    ]

    return pd.Series(values).dropna()

## 17. Identify P&L and balance-sheet tables

In [ ]:
def get_profit_loss_table(symbol):
    tables = get_screener_tables(symbol)

    return find_table_by_row(
        tables,
        ["Sales", "Expenses"]
    )


def get_balance_sheet_table(symbol):
    tables = get_screener_tables(symbol)

    return find_table_by_row(
        tables,
        ["Equity", "Assets"]
    )

## 18. Inspect target P&L and balance sheet

In [ ]:
target_pl = get_profit_loss_table(target_screener_symbol)

print("TARGET P&L")
print("=" * 80)

if target_pl is None:
    print("Could not automatically identify the P&L table.")
else:
    display(target_pl)


target_bs = get_balance_sheet_table(target_screener_symbol)

print("\nTARGET BALANCE SHEET")
print("=" * 80)

if target_bs is None:
    print("Could not automatically identify the balance sheet table.")
else:
    display(target_bs)

## 19. LTM helper

For quarterly data:

\[
LTM = Q_{latest} + Q_{-1} + Q_{-2} + Q_{-3}
\]

**Validate the source's column chronology first.**

In [ ]:
def calculate_ltm(quarterly_values):
    if quarterly_values is None:
        return np.nan

    quarterly_values = pd.Series(quarterly_values).dropna()

    if len(quarterly_values) < 4:
        return np.nan

    # This assumes the four latest observations are the final four values.
    # Change to .iloc[:4] if the source places the newest quarter first.
    return quarterly_values.iloc[-4:].sum()

## 20. Core valuation formulas

**EBITDA**

\[
EBITDA = EBIT + Depreciation
\]

**Enterprise Value**

\[
EV = Market\ Cap + Debt + Minority\ Interest - Cash
\]

**EV / EBITDA**

\[
EV/EBITDA = EV / LTM\ EBITDA
\]

**P/E**

\[
P/E = Market\ Cap / LTM\ Net\ Income
\]

**P/B**

\[
P/B = Market\ Cap / Book\ Value
\]

In [ ]:
def calculate_enterprise_value(
    market_cap,
    total_debt,
    cash,
    minority_interest=0
):
    if pd.isna(market_cap):
        return np.nan

    total_debt = 0 if pd.isna(total_debt) else total_debt
    cash = 0 if pd.isna(cash) else cash
    minority_interest = (
        0 if pd.isna(minority_interest)
        else minority_interest
    )

    return (
        market_cap
        + total_debt
        + minority_interest
        - cash
    )


def calculate_ev_ebitda(enterprise_value, ltm_ebitda):
    if (
        pd.isna(enterprise_value)
        or pd.isna(ltm_ebitda)
        or ltm_ebitda <= 0
    ):
        return np.nan

    return enterprise_value / ltm_ebitda


def calculate_pe(market_cap, ltm_net_income):
    if (
        pd.isna(market_cap)
        or pd.isna(ltm_net_income)
        or ltm_net_income <= 0
    ):
        return np.nan

    return market_cap / ltm_net_income


def calculate_pb(market_cap, book_value):
    if (
        pd.isna(market_cap)
        or pd.isna(book_value)
        or book_value <= 0
    ):
        return np.nan

    return market_cap / book_value

## 21. Financial-data extraction function

In [ ]:
def get_financial_data(ticker_symbol):
    symbol = ticker_symbol.replace(".NS", "").upper()
    print(f"Processing financials: {symbol}")
    tables = get_screener_tables(symbol)
    if len(tables) == 0: return None

    pl_table = find_table_by_row(tables, ["Sales", "Expenses"])
    bs_table = find_table_by_row(tables, ["Equity", "Assets"])

    result = {"Ticker": ticker_symbol, "Sales": np.nan, "EBIT": np.nan, "Depreciation": np.nan, "Net Income": np.nan, "Book Value": np.nan, "Debt": np.nan, "Cash": np.nan}

    if pl_table is not None:
        # Screener uses 'Operating Profit' for EBIT
        result["EBIT"] = row_to_numeric(extract_row(pl_table, "Operating Profit")).iloc[-1] if extract_row(pl_table, "Operating Profit") is not None else np.nan
        result["Depreciation"] = row_to_numeric(extract_row(pl_table, "Depreciation")).iloc[-1] if extract_row(pl_table, "Depreciation") is not None else np.nan
        result["Net Income"] = row_to_numeric(extract_row(pl_table, "Net Profit")).iloc[-1] if extract_row(pl_table, "Net Profit") is not None else np.nan

    if bs_table is not None:
        result["Debt"] = row_to_numeric(extract_row(bs_table, "Borrowings")).iloc[-1] if extract_row(bs_table, "Borrowings") is not None else np.nan
        # Search for Cash or Cash Equivalents
        cash_row = extract_row(bs_table, "Cash") or extract_row(bs_table, "Cash Equivalents")
        result["Cash"] = row_to_numeric(cash_row).iloc[-1] if cash_row is not None else 0.0
        result["Book Value"] = row_to_numeric(extract_row(bs_table, "Equity")).iloc[-1] if extract_row(bs_table, "Equity") is not None else np.nan

    return result

## 22. Download financial data for target + peers

In [ ]:
financial_data = []

print("Downloading financial statements...\n")

for ticker in analysis_tickers:

    data = get_financial_data(ticker)

    if data is not None:
        financial_data.append(data)

    time.sleep(0.5)

financial_df = pd.DataFrame(financial_data)

print("\n✓ Financial data download complete.")
display(financial_df)

## 23. Merge market and financial data

In [ ]:
# 1. Repair market_df before merging to ensure no NaNs for crucial fields
print("Verifying market data for all analysis tickers...")
for idx, row in market_df.iterrows():
    if pd.isna(row["Market Cap"]) or pd.isna(row["Shares Outstanding"]):
        ticker = row["Ticker"]
        print(f"Patching missing market data for {ticker}...")
        robust_data = get_company_info(ticker)
        market_df.at[idx, "Market Cap"] = robust_data.get("Market Cap")
        market_df.at[idx, "Shares Outstanding"] = robust_data.get("Shares Outstanding")

# 2. Merge the repaired market data with financial data
comps_df = pd.merge(
    market_df,
    financial_df,
    on="Ticker",
    how="left"
)

print("\n✓ Merge complete with patched data.")
display(comps_df)

## 24. Calculate EBITDA

In [ ]:
comps_df["EBITDA"] = (
    comps_df["EBIT"]
    + comps_df["Depreciation"]
)

display(
    comps_df[
        ["Company", "EBIT", "Depreciation", "EBITDA"]
    ]
)

## 25. Calculate Enterprise Value

In [ ]:
comps_df["Enterprise Value"] = (
    comps_df.apply(
        lambda row: calculate_enterprise_value(
            row["Market Cap"],
            row["Debt"],
            row["Cash"]
        ),
        axis=1
    )
)

display(
    comps_df[
        [
            "Company",
            "Market Cap",
            "Debt",
            "Cash",
            "Enterprise Value"
        ]
    ]
)

## 26. Calculate trading multiples

In [ ]:
# Treat missing Cash as 0 to ensure Enterprise Value doesn't become NaN
comps_df['Cash'] = comps_df['Cash'].fillna(0)

# Re-calculate Enterprise Value with cleaned Cash data
comps_df["Enterprise Value"] = (
    comps_df.apply(
        lambda row: calculate_enterprise_value(
            row["Market Cap"],
            row["Debt"],
            row["Cash"]
        ),
        axis=1
    )
)

# Allow negative multiples in the raw calculation to identify where the data gaps or losses are
comps_df["EV / EBITDA"] = comps_df.apply(
    lambda row: row["Enterprise Value"] / row["EBITDA"] if row["EBITDA"] != 0 and not pd.isna(row["EBITDA"]) else np.nan,
    axis=1
)

comps_df["P / E"] = comps_df.apply(
    lambda row: row["Market Cap"] / row["Net Income"] if row["Net Income"] != 0 and not pd.isna(row["Net Income"]) else np.nan,
    axis=1
)

comps_df["P / B"] = comps_df.apply(
    lambda row: row["Market Cap"] / row["Book Value"] if row["Book Value"] > 0 and not pd.isna(row["Book Value"]) else np.nan,
    axis=1
)

print("Valuation Multiples (Raw Results including Negatives):")
display(
    comps_df[
        ["Company", "Market Cap", "EBITDA", "Net Income", "EV / EBITDA", "P / E", "P / B"]
    ]
)

## 27. Clean economically meaningless multiples

In [ ]:
import pandas as pd
import numpy as np

# 1. Merge repaired market data with financial data
comps_df = pd.merge(market_df, financial_df, on='Ticker', how='left')

# 2. Perform raw valuation calculations
comps_df['Cash'] = comps_df['Cash'].fillna(0)
comps_df['EBITDA'] = comps_df['EBIT'].fillna(0) + comps_df['Depreciation'].fillna(0)

comps_df['Enterprise Value'] = comps_df.apply(
    lambda row: calculate_enterprise_value(row['Market Cap'], row['Debt'], row['Cash']),
    axis=1
)

comps_df['EV / EBITDA'] = comps_df.apply(
    lambda row: row['Enterprise Value'] / row['EBITDA'] if row['EBITDA'] != 0 and not pd.isna(row['EBITDA']) else np.nan,
    axis=1
)

comps_df['P / E'] = comps_df.apply(
    lambda row: row['Market Cap'] / row['Net Income'] if row['Net Income'] != 0 and not pd.isna(row['Net Income']) else np.nan,
    axis=1
)

comps_df['P / B'] = comps_df.apply(
    lambda row: row['Market Cap'] / row['Book Value'] if row['Book Value'] > 0 and not pd.isna(row['Book Value']) else np.nan,
    axis=1
)

def clean_valuation_multiples(df):
    # Keeping actual raw values (including negatives) as requested
    return df.copy()

# 3. Apply cleaning logic and display
comps_df = clean_valuation_multiples(comps_df)

print("Actual Valuation Results (Raw Multiples):")
display(comps_df[['Company', 'EBITDA', 'Net Income', 'EV / EBITDA', 'P / E', 'P / B']])

## 28. Separate target and comparable companies

In [ ]:
# Identify target from the main comps_df
target_rows = comps_df[comps_df['Ticker'] == target_ticker]
if target_rows.empty:
    raise ValueError(f'Target company {target_ticker} missing from dataset.')
target_row = target_rows.iloc[0]

# Filter peers: Only keep those with positive multiples for all key ratios
peer_comps_df = comps_df[
    (comps_df['Ticker'] != target_ticker) &
    (comps_df['EV / EBITDA'] > 0) &
    (comps_df['P / E'] > 0) &
    (comps_df['P / B'] > 0)
].copy()

print(f"Target: {target_row['Company']}")
print(f"Number of positive-multiple peers found: {len(peer_comps_df)}")
display(peer_comps_df[['Company', 'EV / EBITDA', 'P / E', 'P / B']])

## 29. Calculate peer statistics

In [ ]:
multiple_columns = [
    "EV / EBITDA",
    "P / E",
    "P / B"
]

peer_statistics = pd.DataFrame({
    "Minimum": peer_comps_df[multiple_columns].min(),
    "25th Percentile": peer_comps_df[multiple_columns].quantile(0.25),
    "Median": peer_comps_df[multiple_columns].median(),
    "Mean": peer_comps_df[multiple_columns].mean(),
    "75th Percentile": peer_comps_df[multiple_columns].quantile(0.75),
    "Maximum": peer_comps_df[multiple_columns].max()
})

print("=" * 70)
print("PEER VALUATION STATISTICS")
print("=" * 70)

display(peer_statistics)

## 30. Target valuation functions

In [ ]:
def implied_price_from_ev_ebitda(target_row, multiple):

    if (
        pd.isna(target_row["EBITDA"])
        or pd.isna(target_row["Shares Outstanding"])
        or target_row["Shares Outstanding"] <= 0
        or pd.isna(multiple)
    ):
        return np.nan

    implied_ev = target_row["EBITDA"] * multiple

    implied_equity_value = (
        implied_ev
        - target_row["Debt"]
        + target_row["Cash"]
    )

    return (
        implied_equity_value
        / target_row["Shares Outstanding"]
    )


def implied_price_from_pe(target_row, multiple):

    if (
        pd.isna(target_row["Net Income"])
        or pd.isna(target_row["Shares Outstanding"])
        or target_row["Shares Outstanding"] <= 0
        or pd.isna(multiple)
    ):
        return np.nan

    implied_market_cap = (
        target_row["Net Income"] * multiple
    )

    return (
        implied_market_cap
        / target_row["Shares Outstanding"]
    )


def implied_price_from_pb(target_row, multiple):

    if (
        pd.isna(target_row["Book Value"])
        or pd.isna(target_row["Shares Outstanding"])
        or target_row["Shares Outstanding"] <= 0
        or pd.isna(multiple)
    ):
        return np.nan

    implied_market_cap = (
        target_row["Book Value"] * multiple
    )

    return (
        implied_market_cap
        / target_row["Shares Outstanding"]
    )

## 31. Calculate implied target share-price ranges

In [ ]:
ev_ebitda_low = implied_price_from_ev_ebitda(
    target_row,
    peer_statistics.loc["EV / EBITDA", "25th Percentile"]
)

ev_ebitda_median = implied_price_from_ev_ebitda(
    target_row,
    peer_statistics.loc["EV / EBITDA", "Median"]
)

ev_ebitda_high = implied_price_from_ev_ebitda(
    target_row,
    peer_statistics.loc["EV / EBITDA", "75th Percentile"]
)


pe_low = implied_price_from_pe(
    target_row,
    peer_statistics.loc["P / E", "25th Percentile"]
)

pe_median = implied_price_from_pe(
    target_row,
    peer_statistics.loc["P / E", "Median"]
)

pe_high = implied_price_from_pe(
    target_row,
    peer_statistics.loc["P / E", "75th Percentile"]
)


pb_low = implied_price_from_pb(
    target_row,
    peer_statistics.loc["P / B", "25th Percentile"]
)

pb_median = implied_price_from_pb(
    target_row,
    peer_statistics.loc["P / B", "Median"]
)

pb_high = implied_price_from_pb(
    target_row,
    peer_statistics.loc["P / B", "75th Percentile"]
)


valuation_results = pd.DataFrame({
    "Method": ["EV / EBITDA", "P / E", "P / B"],
    "Low": [ev_ebitda_low, pe_low, pb_low],
    "Median": [ev_ebitda_median, pe_median, pb_median],
    "High": [ev_ebitda_high, pe_high, pb_high]
})

display(valuation_results)

## 32. Retrieve the target's current price and 52-week range

In [ ]:
current_price = target_row["Current Price"]
fifty_two_week_low = target_row["52W Low"]
fifty_two_week_high = target_row["52W High"]

print(f"Current Price : ₹{current_price:,.2f}")
print(f"52W Low       : ₹{fifty_two_week_low:,.2f}")
print(f"52W High      : ₹{fifty_two_week_high:,.2f}")

## 33. Prepare football-field valuation ranges

In [ ]:
football_field_data = {
    "52-Week Trading Range": (
        fifty_two_week_low,
        fifty_two_week_high
    ),
    "EV / EBITDA": (
        ev_ebitda_low,
        ev_ebitda_high
    ),
    "P / E": (
        pe_low,
        pe_high
    ),
    "P / B": (
        pb_low,
        pb_high
    )
}

football_field_data

## 34. Generate the IB-style football field

In [ ]:
def create_football_field(
    valuation_ranges,
    current_price,
    company_name
):
    # Filter only for NaNs, allow negative values as requested by user
    valid_ranges = {}
    for label, values in valuation_ranges.items():
        low, high = values
        if pd.notna(low) and pd.notna(high):
            valid_ranges[label] = (low, high)

    if len(valid_ranges) == 0:
        print("No valid valuation ranges available.")
        return

    labels = list(valid_ranges.keys())
    fig, ax = plt.subplots(figsize=(13, 7))
    y_positions = np.arange(len(labels))

    for i, label in enumerate(labels):
        low, high = valid_ranges[label]

        # Plot the range bar
        ax.plot(
            [low, high],
            [i, i],
            linewidth=12,
            solid_capstyle="round"
        )

        ax.scatter([low, high], [i, i], s=60)

        # Label the low and high values
        ax.text(low, i + 0.18, f"₹{low:,.0f}", ha="center", fontsize=9)
        ax.text(high, i + 0.18, f"₹{high:,.0f}", ha="center", fontsize=9)

    if pd.notna(current_price):
        ax.axvline(
            current_price,
            linestyle="--",
            linewidth=2,
            color='red',
            label=f"Current Price: ₹{current_price:,.0f}"
        )

    ax.set_yticks(y_positions)
    ax.set_yticklabels(labels)
    ax.set_xlabel("Implied Share Price (₹)", fontsize=11)
    ax.set_title(f"{company_name} — Public Comparables Valuation", fontsize=15, fontweight="bold")
    ax.grid(axis="x", alpha=0.25)
    ax.legend()

    plt.tight_layout()
    plt.show()

create_football_field(
    football_field_data,
    current_price,
    target_row["Company"]
)

## 35. Create final comps table

In [ ]:
final_columns = [
    "Company",
    "Ticker",
    "Current Price",
    "Market Cap",
    "Enterprise Value",
    "EBITDA",
    "Net Income",
    "Book Value",
    "Debt",
    "Cash",
    "EV / EBITDA",
    "P / E",
    "P / B",
    "52W Low",
    "52W High"
]

final_comps = comps_df[final_columns].copy()

final_comps.insert(
    1,
    "Role",
    np.where(
        final_comps["Ticker"] == target_ticker,
        "Target",
        "Comparable"
    )
)

display(final_comps)

## 36. Data-quality checks

In [ ]:
def check_data_quality(row):

    issues = []

    if pd.isna(row["Current Price"]):
        issues.append("Missing price")

    if pd.isna(row["Market Cap"]):
        issues.append("Missing market cap")

    if pd.isna(row["Enterprise Value"]):
        issues.append("Missing EV")

    if pd.isna(row["EBITDA"]):
        issues.append("Missing EBITDA")

    if pd.isna(row["Net Income"]):
        issues.append("Missing net income")

    if pd.isna(row["Book Value"]):
        issues.append("Missing book value")

    if len(issues) == 0:
        return "PASS"

    return " | ".join(issues)


final_comps["Data Quality"] = final_comps.apply(
    check_data_quality,
    axis=1
)

display(
    final_comps[
        ["Company", "Role", "Data Quality"]
    ]
)

## 37. Final valuation summary

In [ ]:
print("=" * 80)
print("VALUATION SUMMARY REPORT".center(80))
print("=" * 80)

# Prepare target info table
target_summary = pd.DataFrame({
    "Metric": ["Target Company", "Sector", "Industry", "Current Price"],
    "Value": [
        target_row['Company'],
        target_info['Sector'],
        target_info['Industry'],
        f"₹{current_price:,.2f}"
    ]
})

print("\n[ TARGET PROFILE ]")
display(target_summary)

# Format the valuation results for better readability
formatted_valuation = valuation_results.copy()
for col in ['Low', 'Median', 'High']:
    formatted_valuation[col] = formatted_valuation[col].apply(lambda x: f"₹{x:,.0f}")

print("\n[ IMPLIED SHARE PRICE RANGES ]")
display(formatted_valuation)

print("\n" + "=" * 80)

## 38. Export the complete analysis to Excel

In [ ]:
output_file = (
    "Automated_Public_Comps_"
    + target_ticker.replace(".NS", "")
    + ".xlsx"
)

with pd.ExcelWriter(
    output_file,
    engine="openpyxl"
) as writer:

    final_comps.to_excel(
        writer,
        sheet_name="Trading Comps",
        index=False
    )

    peer_statistics.to_excel(
        writer,
        sheet_name="Peer Statistics"
    )

    valuation_results.to_excel(
        writer,
        sheet_name="Target Valuation",
        index=False
    )

    peers_df.to_excel(
        writer,
        sheet_name="Peer Group",
        index=False
    )

    pd.DataFrame(
        [target_info]
    ).to_excel(
        writer,
        sheet_name="Target Information",
        index=False
    )

print(f"✓ Excel file created: {output_file}")

## 39. Download the Excel output

In [ ]:
from google.colab import files

files.download(output_file)